# GD Arena — ready model + voice UI
Upload this notebook to Google Colab and select **Runtime → Run all** using a normal CPU runtime.
The saved model and complete app are embedded below; **no training, ZIP upload, API key or GPU is needed**.

The first run installs Python libraries and downloads a pinned 75 MB pretrained Whisper speech model. The stance model is already embedded. The app appears inside the notebook after startup.
Keep this runtime connected. Use the local Chrome/Edge app if the embedded browser restricts microphone or playback.

**Project scope:** a real English stance classifier + prepared discussion partners + local/server Whisper transcription and browser speech playback.
Hindi/Hinglish speech needs a manually reviewed English version. Public test accuracy: 43.31%; authored GD test: 63.33%.
This is an experimental baseline, not an interview scoring system.

In [ ]:
# 1. Extract the embedded, already-trained app. No model training happens here.
from pathlib import Path
import base64, hashlib, io, json, os, socket, subprocess, sys, time, urllib.request, zipfile

PROJECT = (Path.cwd() / 'gd_arena_ready_demo').resolve()
PROJECT.mkdir(parents=True, exist_ok=True)
PAYLOAD = (
    '...'
)
assert hashlib.sha256(base64.b64decode(PAYLOAD)).hexdigest() == '8c8240e7d2b2941a5ab017d0db1537541092a347f362ea4a25b9319939f671c5'

with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    for entry in archive.infolist():
        destination = (PROJECT / entry.filename).resolve()
        if not destination.is_relative_to(PROJECT):
            raise ValueError('Unsafe archive path')
    archive.extractall(PROJECT)
print('Saved model and app extracted. No training is needed.')

In [ ]:
# 2. Install packages into this demo folder, isolated from notebook imports.
PACKAGES = PROJECT / '.runtime' / 'packages'
PACKAGES.mkdir(parents=True, exist_ok=True)
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
    '--target', str(PACKAGES), '--upgrade', '-r', str(PROJECT / 'requirements.txt')
])
print('Demo libraries are ready.')

In [ ]:
# 3. Start the Python model server in a separate process.
# A fresh process uses the exact installed versions without a kernel restart.
if 'gd_process' in globals() and gd_process.poll() is None:
    gd_process.terminate()
    gd_process.wait(timeout=15)
if 'gd_log' in globals():
    gd_log.close()
with socket.socket() as available_port:
    available_port.bind(('127.0.0.1', 0))
    PORT = available_port.getsockname()[1]
env = os.environ.copy()
env['PYTHONPATH'] = str(PACKAGES)
env['PYTHONUNBUFFERED'] = '1'
subprocess.check_call([sys.executable, str(PROJECT / 'setup_voice.py')], cwd=PROJECT, env=env)
gd_log = (PROJECT / 'demo_server.log').open('w', encoding='utf-8')
gd_process = subprocess.Popen([sys.executable, str(PROJECT / 'app.py'), '--port', str(PORT)],
                              cwd=PROJECT, env=env, stdout=gd_log, stderr=subprocess.STDOUT)
for attempt in range(120):
    if gd_process.poll() is not None:
        gd_log.flush()
        raise RuntimeError((PROJECT / 'demo_server.log').read_text())
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/health', timeout=1) as response:
            health = json.load(response)
        if health.get('model_loaded'):
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Startup timed out. Check demo_server.log and rerun this cell.')
print('GD Arena is running. Choose a topic in the interface below.')

In [ ]:
# 4. Open the live UI inside Colab (the supported iframe helper).
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=1050)
except ImportError:
    from IPython.display import IFrame, display
    print(f'Local demo: http://127.0.0.1:{PORT}/')
    display(IFrame(f'http://127.0.0.1:{PORT}/', width='100%', height=1050))

## Demonstrate the model
1. Pick a topic and session length, then enter the practice room.
2. Press Speak, then pause for 2 seconds when finished. English microphone turns share automatically. Switch off Auto share to review first. Hindi/Hinglish still needs English review; uploads stay editable.
3. Show the model's stance prediction and a prepared partner perspective.
4. End the session and download the transcript.

For Hindi/Hinglish, review the speech transcript and manually write its English meaning.
If speech is blocked in the Colab iframe, use typed input here and the local app for the voice demo.
The notebook payload/server was checked locally; the Google-account iframe is environment-dependent.

To retrain and show feature engineering, dataset splits and evaluation, use the separate **GD_Arena_Colab.ipynb** training notebook.